In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;
USE DATABASE AIRLINE_DB;

In [ ]:
%%sql
SELECT *
FROM AIRLINE_DB.RAW.AIRLINE_RAW
LIMIT 20;

In [ ]:
%%sql
SELECT * 
FROM AIRLINE_DB.AUDIT.LOG;

In [ ]:
%%sql
SELECT *
FROM AIRLINE_DB.STAGE.AIRLINE_STAGE
LIMIT 20;

In [ ]:
%%sql
SELECT * 
FROM AIRLINE_DB.DWH.FACT_FLIGHTS
LIMIT 20;

In [ ]:
%%sql
DROP TABLE AIRLINE_DB.DWH.FACT_FLIGHTS;

UNDROP TABLE AIRLINE_DB.DWH.FACT_FLIGHTS;

SELECT * 
FROM AIRLINE_DB.DWH.FACT_FLIGHTS
LIMIT 20; 

In [ ]:
%%sql
CREATE OR REPLACE TABLE AIRLINE_DB.DWH.FACT_FLIGHTS_BACKUP 
CLONE AIRLINE_DB.DWH.FACT_FLIGHTS AT (OFFSET => -30); --30 s minute

SELECT * FROM AIRLINE_DB.DWH.FACT_FLIGHTS_BACKUP 
LIMIT 10;

In [ ]:
%%sql
-- The DML Time Travel Query: Read data from the past
SELECT COUNT(*) AS HISTORIC_ROW_COUT 
FROM AIRLINE_DB.STAGE.AIRLINE_STAGE 
AT (OFFSET => -30);

In [ ]:
%%sql
-- Setup: Someone accidentally deletes a specific row
DELETE FROM AIRLINE_DB.DWH.DIM_PASSENGER WHERE GENDER = 'FEMALE';

In [ ]:
%%sql
-- The DML Time Travel Query: Pull just that single row from 30 s ago and put it back
INSERT INTO AIRLINE_DB.DWH.DIM_PASSENGER 
SELECT * FROM AIRLINE_DB.DWH.DIM_PASSENGER AT (OFFSET => -30) 
WHERE GENDER = 'FEMALE';

In [ ]:
CREATE ROW ACCESS POLICY IF NOT EXISTS AIRLINE_DB.DWH.FLIGHT_REGION_POLICY
AS (continent_val VARCHAR) RETURNS BOOLEAN ->
    CASE
        -- Admins see everything
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN', 'DATA_ENGINEER') THEN TRUE
        
        -- EU Managers only see European flights
        WHEN CURRENT_ROLE() = 'EU_MANAGER' AND continent_val = 'EUROPE' THEN TRUE

        -- US Managers only see North American flights
        WHEN CURRENT_ROLE() = 'NA_MANAGER' AND continent_val = 'NORTH AMERICA' THEN TRUE
        
        -- Everyone else sees nothing
        ELSE FALSE
END;

In [ ]:
%%sql
CREATE OR REPLACE SECURE VIEW AIRLINE_DB.DWH.SECURE_REGION_VIEW AS
SELECT DP.FIRST_NAME, DP.LAST_NAME, DP.GENDER, DP.AGE, DP.NATIONALITY, 
    DA.AIRPORT_CONTINENT, DA.CONTINENTS, DD.FLIGHT_YEAR, DFD.FLIGHT_STATUS,
    DFD.TICKET_TYPE, DFD.PASSENGER_STATUS
FROM AIRLINE_DB.DWH.FACT_FLIGHTS AS FF
    JOIN AIRLINE_DB.DWH.DIM_PASSENGER AS DP USING (PASSENGER_SK)
    JOIN AIRLINE_DB.DWH.DIM_AIRPORT AS DA USING (AIRPORT_ID) 
    JOIN AIRLINE_DB.DWH.DIM_DATE AS DD ON DD.CALENDAR_DATE = FF.DEPARTURE_DATE
    JOIN AIRLINE_DB.DWH.DIM_FLIGHT_DETAILS AS DFD USING (DETAILS_KEY);
    

ALTER VIEW AIRLINE_DB.DWH.SECURE_REGION_VIEW
ADD ROW ACCESS POLICY AIRLINE_DB.DWH.FLIGHT_REGION_POLICY ON (CONTINENTS);

In [ ]:
%%sql
SELECT *
FROM AIRLINE_DB.DWH.SECURE_REGION_VIEW
LIMIT 20;

In [ ]:
%%sql
USE ROLE SECURITYADMIN;

-- Create roles
CREATE ROLE IF NOT EXISTS EU_MANAGER;
CREATE ROLE IF NOT EXISTS NA_MANAGER;

-- Grant access from Secure View to roles
GRANT SELECT ON VIEW AIRlINE_DB.DWH.SECURE_REGION_VIEW TO ROLE EU_MANAGER;
GRANT SELECT ON VIEW AIRlINE_DB.DWH.SECURE_REGION_VIEW TO ROLE NA_MANAGER;

-- Define role to user
GRANT ROLE EU_MANAGER TO USER VKRYLOVA;
GRANT ROLE NA_MANAGER TO USER VKRYLOVA;

In [ ]:
%%sql
USE ROLE EU_MANAGER;

SELECT * 
FROM AIRLINE_DB.DWH.SECURE_REGION_VIEW
LIMIT 20;

In [ ]:
%%sql
USE ROLE NA_MANAGER;

SELECT * FROM AIRLINE_DB.DWH.SECURE_REGION_VIEW
LIMIT 20;